In [1]:
options(warn = -1, stringsAsFactors = FALSE)
packages <- c("Seurat", "SeuratObject", "Matrix", "dplyr", "fields", "MASS","scDesign3", "SingleCellExperiment", "tidyverse", 
              "CARD", "spacexr", "SPOTlight", "CellTrek","SpatialDecon", "tibble", "tidyr", "ggplot2", "ggpubr", "patchwork", 
              "RColorBrewer", "scales", "reshape2", "reticulate", "cowplot", "viridis", "readr", "stringr", "magrittr")
invisible(suppressPackageStartupMessages({lapply(packages, library, character.only = TRUE, quietly = TRUE, warn.conflicts = FALSE)}))

#### **Loading data**

In [2]:
# Load reference
reference <- readRDS("../Human glioblastoma/1_scRNAseq/8167_Annotated.rds")
if (inherits(reference, "Seurat")) {
  sce <- as.SingleCellExperiment(reference, assay = "RNA")
} else if (inherits(reference, "SingleCellExperiment")) {
  sce <- reference
} else {
  stop("Reference object is neither Seurat nor SingleCellExperiment.")
}

## Settings
time_col <- "orig.ident"
cell_type_col <- "cell_types"
target_celltypes <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")

## Filter cells and prepare SCE
meta <- as.data.frame(colData(sce))
keep <- !is.na(meta[[cell_type_col]]) & !is.na(meta[[time_col]]) &  meta[[cell_type_col]] %in% target_celltypes & meta[[time_col]] %in% c("Tumor", "2wk", "24wk")
sce <- sce[, keep]

## QC
sce <- sce[, colSums(assay(sce, "counts")) >= 200]

## Clean metadata, Map to TF0, TF1, TF2, Numeric variable for pseudotime
colData(sce)[[cell_type_col]] <- factor(colData(sce)[[cell_type_col]], levels = target_celltypes)
colData(sce)$time_numeric <- dplyr::recode(colData(sce)[[time_col]], "Tumor" = 0, "2wk" = 1, "24wk" = 2)
colData(sce)$time_numeric <- as.numeric(colData(sce)$time_numeric)

## Select top variable genes
cat("\n=== GENE FILTERING ===\n")
counts_mat <- assay(sce, "counts")
cat(sprintf("Before: %d genes\n", nrow(counts_mat)))

detection_rate <- rowSums(counts_mat > 0) / ncol(counts_mat)
counts_mat <- counts_mat[detection_rate >= 0.05, ]
cat(sprintf("After detection rate: %d genes\n", nrow(counts_mat)))

gene_means <- rowMeans(counts_mat)
counts_mat <- counts_mat[gene_means > 0.05, ]
cat(sprintf("After mean: %d genes\n", nrow(counts_mat)))

gene_vars <- apply(counts_mat, 1, var)
top_genes <- names(sort(gene_vars, decreasing = TRUE))[1:min(1000, nrow(counts_mat))]
cat(sprintf("Top genes: %d genes\n", length(top_genes)))

sce <- sce[top_genes, ]
cat(sprintf("SCE now has: %d genes\n\n", nrow(sce)))

# Sampling
cat("=== SAMPLING ===\n")
counts_matrix <- as.matrix(assay(sce, "counts"))
meta_df <- as.data.frame(colData(sce))
cat(sprintf("Input: %d genes, %d cells\n", nrow(counts_matrix), ncol(counts_matrix)))

dist_matrix <- as.matrix(table(meta_df[[cell_type_col]], meta_df[[time_col]]))
group_props <- dist_matrix / sum(dist_matrix)

set.seed(123)
selected_cells <- c()

for(ct in rownames(dist_matrix)) {
    for(tp in colnames(dist_matrix)) {
        group_cells <- rownames(meta_df)[meta_df[[cell_type_col]] == ct & meta_df[[time_col]] == tp]
        n_avail <- length(group_cells)
        if(n_avail == 0) next
        n_sample <- min(n_avail, max(100, round(group_props[ct, tp] * 8000)))
        selected_cells <- c(selected_cells, sample(group_cells, n_sample))
    }
}

RefSCE <- SingleCellExperiment(assays = list(counts = counts_matrix[, selected_cells, drop = FALSE]), colData = meta_df[selected_cells, , drop = FALSE])
colData(RefSCE)[[cell_type_col]] <- factor(colData(RefSCE)[[cell_type_col]], levels = target_celltypes)

cat(sprintf("Output: %d genes, %d cells\n", nrow(RefSCE), ncol(RefSCE)))
zero_fraction <- sum(assay(RefSCE, "counts") == 0) / (nrow(RefSCE) * ncol(RefSCE))
cat(sprintf("Zero fraction: %.4f\n", zero_fraction))

Final_outputs <- "Final_outputs"
dir.create(Final_outputs, showWarnings = FALSE, recursive = TRUE)

saveRDS(RefSCE, file = paste0(Final_outputs, "/ProcessedSCE.rds"), compress = TRUE)
cat("\nReference SCE data is ready to be processed.")


=== GENE FILTERING ===
Before: 25065 genes
After detection rate: 3824 genes
After mean: 3824 genes
Top genes: 1000 genes
SCE now has: 1000 genes

=== SAMPLING ===
Input: 1000 genes, 24368 cells
Output: 1000 genes, 8023 cells
Zero fraction: 0.7152

Reference SCE data is ready to be processed.

#### **Simulation**

In [2]:
## Settings and loading data
Final_outputs <- "Final_outputs"
time_levels <- c("Tumor", "2wk", "24wk")
target_celltypes <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")

###### *Generate simulated single-cell data*

In [5]:
RefSCE <- readRDS("./Final_outputs/ProcessedSCE.rds")
set.seed(123)
simu_res <- scdesign3(sce = RefSCE, assay_use = "counts",  celltype = cell_type_col, pseudotime = "time_numeric",
                      spatial = NULL, other_covariates = NULL, mu_formula = "cell_types + time_numeric", sigma_formula = "cell_types",
                      family_use = "nb", n_cores = 1, ncell = 5000, correlation_function = "default", corr_formula = "cell_types",
                      empirical_quantile = FALSE, usebam = FALSE, edf_flexible = FALSE, copula = "gaussian", fastmvn = TRUE, 
                      if_sparse = FALSE, simplify = FALSE, DT = FALSE, pseudo_obs = FALSE, family_set = "gauss", nonnegative = TRUE, 
                      return_model = TRUE, nonzerovar = FALSE, parallelization = "lapply", trace = FALSE)   

## Build metadata
sim_meta <- as.data.frame(simu_res$new_covariate)

time_rounded <- round(sim_meta$time_numeric)
time_rounded[time_rounded < 0] <- 0
time_rounded[time_rounded > 2] <- 2
colnames(sim_meta)[colnames(sim_meta) == "time_numeric"] <- "time_continuous"
sim_meta$orig.ident <- time_levels[time_rounded + 1]
sim_meta$time_numeric <- time_rounded

sim_meta$corr_group <- NULL
sim_meta$time_continuous <- NULL

sim_meta$cell_types <- factor(sim_meta$cell_types, levels = target_celltypes)
sim_meta$orig.ident <- factor(sim_meta$orig.ident, levels = time_levels)

SimulatedSC <- CreateSeuratObject(counts = simu_res$new_count, meta.data = sim_meta, project = "SimulatedSC")

saveRDS(SimulatedSC, file = file.path(Final_outputs, paste0("SimulatedSC.rds")))
cat("Simulated single cell data was successfully generated.")

Input Data Construction Start

Input Data Construction End

Start Marginal Fitting

Marginal Fitting End

Start Copula Fitting

Convert Residuals to Multivariate Gaussian

Converting End

Copula group Astrocytes starts

Copula group Epithelial cells starts

Copula group Neutrophils starts

Copula group OPCs starts

Copula group Endothelial cells starts

Copula group Proliferation starts

Copula Fitting End

Start Parameter Extraction

Parameter
Extraction End

Start Generate New Data

Use Copula to sample a multivariate quantile matrix

Sample Copula group Astrocytes starts

Sample Copula group Epithelial cells starts

Sample Copula group Neutrophils starts

Sample Copula group OPCs starts

Sample Copula group Endothelial cells starts

Sample Copula group Proliferation starts

New Data Generating End



Simulated single cell data was successfully generated.

###### *Generate synthetic spatial data*

In [6]:
eps <- 1e-8
field_noise_sd <- 0.25
cells_per_spot <- 5:15
spatial_smoothness <- 0.20

## DATA PREPARATION
# Load simulated scRNA-seq
SimulatedSC <- readRDS(file.path(Final_outputs, "SimulatedSC.rds"))

# Load real spatial (for coordinates)
refSpatial <- readRDS("../Human glioblastoma/2_stRNAseq/Human_Glioblastoma_Whole_SVF.rds")

# Get real coordinates
coords <- GetTissueCoordinates(refSpatial)
coords <- as.data.frame(coords[, c("x", "y")])
n_spots <- nrow(coords)

# Extract simulated data
sim_counts <- GetAssayData(SimulatedSC, assay = "RNA", layer = "counts")
sim_meta <- SimulatedSC@meta.data

# Group cells by time point
cell_groups <- split(colnames(SimulatedSC), sim_meta$orig.ident)

## HELPER: Generate spatial field
generate_spatial_field <- function(coords, smoothness, noise_sd) {
    xy <- as.matrix(coords[, c("x", "y")])
    dmat <- fields::rdist(xy)
    cov_mat <- exp(-dmat / (max(dmat) * smoothness + eps))
    diag(cov_mat) <- diag(cov_mat) + 1e-6
    L <- chol(cov_mat)
    field <- as.numeric(crossprod(L, rnorm(nrow(xy))))
    field <- as.numeric(scale(field)) + rnorm(length(field), mean = 0, sd = noise_sd)
    return(field)
}

## GENERATE SYNTHETIC SPATIAL DATA
spots_coords <- list()
seurat_st <- list()
set.seed(123)

for (tp in time_levels) {
    message("Building: ", tp)  

    tp_cells <- cell_groups[[tp]]
    if (is.null(tp_cells) || length(tp_cells) == 0) {
        message("  ⚠️  No cells for ", tp, ", skipping...")
        next
    }

    message("  Available cells: ", length(tp_cells))

    spot_ids <- paste0(tp, "_spot_", 1:n_spots)

    # Initialize matrices
    spot_counts <- matrix(0, nrow = nrow(sim_counts), ncol = n_spots)
    rownames(spot_counts) <- rownames(sim_counts)
    colnames(spot_counts) <- spot_ids

    spot_props <- matrix(0, nrow = n_spots, ncol = length(target_celltypes))
    colnames(spot_props) <- target_celltypes
    rownames(spot_props) <- spot_ids

    # Calculate base proportions for this time point
    tp_meta <- sim_meta[tp_cells, ]
    base_props <- table(factor(tp_meta$cell_types, levels = target_celltypes))
    base_props <- as.numeric(base_props) / sum(base_props)
    names(base_props) <- target_celltypes

    # Generate spatial fields for each cell type
    message("  Generating spatial fields...")
    field_mat <- sapply(target_celltypes, function(ct) {generate_spatial_field(coords, spatial_smoothness, field_noise_sd)})

    # Combine base proportions with spatial bias
    abundance_mat <- matrix(NA_real_, nrow = n_spots, ncol = length(target_celltypes))
    colnames(abundance_mat) <- target_celltypes

    for (ct in target_celltypes) {
        abundance_mat[, ct] <- exp(log(base_props[ct] + eps) + field_mat[, ct])
    }
    abundance_mat <- abundance_mat / rowSums(abundance_mat)

    # Build spots
    message("  Building ", n_spots, " spots...")
    pb <- txtProgressBar(min = 0, max = n_spots, style = 3)

    for (i in 1:n_spots) {
        n_cells <- sample(cells_per_spot, 1)
        
        # Select cell types based on abundance
        ct_probs <- abundance_mat[i, ]
        selected_cts <- sample(target_celltypes, n_cells, prob = ct_probs, replace = TRUE)
        
        # Select specific cells for each cell type
        selected_cells <- c()
        for (ct_name in unique(selected_cts)) {
            n_ct <- sum(selected_cts == ct_name)
            ct_pool <- tp_cells[tp_meta$cell_types == ct_name]
            if (length(ct_pool) > 0) {
                selected_cells <- c(selected_cells, sample(ct_pool, min(n_ct, length(ct_pool)), replace = TRUE))
            }
        }

        # Fill remaining if needed
        while (length(selected_cells) < n_cells) {selected_cells <- c(selected_cells, sample(tp_cells, 1))}
        
        selected_cells <- selected_cells[1:n_cells]

        # Aggregate counts
        spot_counts[, i] <- rowSums(sim_counts[, selected_cells, drop = FALSE])

        # Ground truth proportions
        ct <- sim_meta[selected_cells, "cell_types"]
        spot_props[i, ] <- table(factor(ct, target_celltypes)) / n_cells
        
        setTxtProgressBar(pb, i)
    }
    close(pb)

    # Store raw data
    spots_coords[[tp]] <- list(coords = coords, spot_ids = spot_ids, counts = spot_counts, proportions = spot_props, abundance = abundance_mat)

    # Create Seurat object
    message("  Creating Seurat object...")

    # Metadata
    meta <- data.frame(spot_id = spot_ids, x = coords$x, y = coords$y, time_point = tp, row.names = spot_ids)

    # Add ground truth proportions
    props_df <- as.data.frame(spot_props)
    colnames(props_df) <- paste0("gt_", colnames(props_df))
    meta <- cbind(meta, props_df)

    # Create Seurat
    obj <- CreateSeuratObject(counts = spot_counts, meta.data = meta, project = tp)
    seurat_st[[tp]] <- obj
}

## SAVE OUTPUTS
saveRDS(seurat_st, file = file.path(Final_outputs, "SimulatedST.rds"))
message("✅ Synthetic spatial data generated.")

Building: Tumor

  Available cells: 1973

  Generating spatial fields...

  Building 3291 spots...



  |======================================================================| 100%


  Creating Seurat object...

Building: 2wk

  Available cells: 2475

  Generating spatial fields...

  Building 3291 spots...



  |======================================================================| 100%


  Creating Seurat object...

Building: 24wk

  Available cells: 552

  Generating spatial fields...

  Building 3291 spots...



  |======================================================================| 100%


  Creating Seurat object...

✅ Synthetic spatial data generated.



###### *Synthetic Spatial Data Validation*

In [8]:
validate_synthetic_spatial <- function(seurat_st) {
    cat("=== Synthetic Spatial Data Validation ===\n\n")

    for (tp in names(seurat_st)) {
        obj <- seurat_st[[tp]]
        gt_cols <- grep("^gt_", colnames(obj@meta.data), value = TRUE)

        cat("---", tp, "---\n")
        cat("  Spots:", ncol(obj), "\n")
        cat("  Genes:", nrow(obj), "\n")
        cat("  Cell types:", length(gt_cols), "\n")
        
        # Proportion sums
        row_sums <- rowSums(obj@meta.data[, gt_cols])
        cat("  Proportion sum range:", min(row_sums), "-", max(row_sums), "\n")
        
        # Check for NA
        na_counts <- sum(is.na(obj@assays$RNA$counts))
        cat("  NA in counts:", na_counts, "\n")
        
        # Check coordinates
        cat("  x range:", range(obj$x), "\n")
        cat("  y range:", range(obj$y), "\n\n")
    }
    cat("✅ Validation complete!\n")
}
validate_synthetic_spatial(seurat_st)

=== Synthetic Spatial Data Validation ===

--- Tumor ---
  Spots: 3291 
  Genes: 1000 
  Cell types: 6 
  Proportion sum range: 1 - 1 
  NA in counts: 0 
  x range: 2430 10354 
  y range: 1442 10673 

--- 2wk ---
  Spots: 3291 
  Genes: 1000 
  Cell types: 6 
  Proportion sum range: 1 - 1 
  NA in counts: 0 
  x range: 2430 10354 
  y range: 1442 10673 

--- 24wk ---
  Spots: 3291 
  Genes: 1000 
  Cell types: 6 
  Proportion sum range: 1 - 1 
  NA in counts: 0 
  x range: 2430 10354 
  y range: 1442 10673 

✅ Validation complete!


#### **Deconvolution**

In [3]:
Final_outputs <- "Final_outputs"
deconv_dir <- "./Final_outputs/Deconvolution"
dir.create(deconv_dir, recursive = TRUE, showWarnings = FALSE)

SimulatedSC <- readRDS("./Final_outputs/SimulatedSC.rds")
SimulatedST <- readRDS("./Final_outputs/SimulatedST.rds")

###### *CARD*

In [3]:
all_CARD_results <- list()                                                          # Store each temporary timepoint result in memory
for (tp in c("Tumor", "2wk", "24wk")) {                                             # Process one timepoint at a time
    message("Running CARD for ", tp)
    reference_tp <- subset(SimulatedSC, subset = orig.ident == tp)                  # Select matching scRNA reference
    spatial_tp <- SimulatedST[[tp]]                                                 # Spatial pseudo-spot object for this timepoint

    sc_count <- GetAssayData(reference_tp, assay = "RNA", layer = "counts")         # Prepare CARD inputs
    spatial_count <- GetAssayData(spatial_tp, assay = "RNA", layer = "counts")
    spatial_location <- spatial_tp@meta.data[, c("x", "y"), drop = FALSE]           # Coordinates required by CARD
    spatial_location <- spatial_location[colnames(spatial_count), , drop = FALSE]   # Ensure coordinate rownames match spatial count columns

    common_genes <- intersect(rownames(sc_count), rownames(spatial_count))          # Keep shared genes only
    sc_count <- sc_count[common_genes, , drop = FALSE]
    spatial_count <- spatial_count[common_genes, , drop = FALSE]

    sc_meta <- reference_tp@meta.data[, c("cell_types"), drop = FALSE]              # Metadata in CARD-compatible format
    sc_meta$cellID <- rownames(sc_meta)
    sc_meta$sampleInfo <- tp
    sc_meta$cellType <- sc_meta$cell_types
    sc_meta <- sc_meta[, c("cellID", "sampleInfo", "cellType")]

    # Run CARD
    invisible(capture.output({
        CARD_obj <- createCARDObject(sc_count, sc_meta, spatial_count, spatial_location, ct.varname = "cellType",
                                     ct.select = unique(sc_meta$cellType), sample.varname = "sampleInfo", minCountGene = 100, minCountSpot = 5)
        CARD_obj <- suppressMessages(CARD_deconvolution(CARD_obj))
    }))

    # Create a benchmark-ready table
    estimated_prop <- as.data.frame(CARD_obj@Proportion_CARD)
    estimated_prop$spot_id <- rownames(estimated_prop)
    
    coordinates <- CARD_obj@spatial_location
    coordinates$spot_id <- rownames(coordinates)

    # Extract true simulated proportions
    gt_cols <- grep("^gt_", colnames(spatial_tp@meta.data), value = TRUE)
    ground_truth <- spatial_tp@meta.data[estimated_prop$spot_id, gt_cols, drop = FALSE]
    ground_truth$spot_id <- rownames(ground_truth)

    # Combine coordinate, estimate, and ground truth
    result_tp <- merge(coordinates, estimated_prop, by = "spot_id", sort = FALSE)
    result_tp <- merge(result_tp, ground_truth, by = "spot_id", sort = FALSE)
    result_tp$timepoint <- tp
    result_tp <- result_tp[, c("timepoint", "spot_id", "x", "y", setdiff(colnames(result_tp), c("timepoint", "spot_id", "x", "y")))]
    all_CARD_results[[tp]] <- result_tp
}

# Save only one combined result
CARD_all_timepoints <- do.call(rbind, all_CARD_results)
rownames(CARD_all_timepoints) <- NULL

saveRDS(CARD_all_timepoints, file = file.path(deconv_dir, "CARD_all_timepoints.rds"), compress = TRUE)
write.csv(CARD_all_timepoints, file = file.path(deconv_dir, "CARD_all_timepoints_prop.csv"), row.names = FALSE)
cat("\n✅ CARD pipeline completed!\n")

Running CARD for Tumor

Running CARD for 2wk

Running CARD for 24wk




✅ CARD pipeline completed!


###### *RCTD*

In [4]:
cell_types <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")
gt_cols <- paste0("gt_", cell_types)

rctd_standardized_results <- list()
rctd_objects <- list()
for (tp in names(SimulatedST)) {
    cat("RCTD Running for", tp, "...\n")
    
    sc_tp <- subset(SimulatedSC, subset = orig.ident == tp)
    if (ncol(sc_tp) == 0) {stop("No scRNA-seq cells found for timepoint: ", tp)}

    counts_sc <- GetAssayData(sc_tp, assay = "RNA", layer = "counts")
    cluster_sc <- as.factor(sc_tp$cell_types)
    levels(cluster_sc) <- gsub("/", "-", levels(cluster_sc))
    cluster_sc <- droplevels(cluster_sc)
    nUMI_sc <- colSums(counts_sc)

    st_obj <- SimulatedST[[tp]]
    counts_st <- GetAssayData(st_obj, assay = "RNA", layer = "counts")
    spot_ids <- colnames(counts_st)
    st_meta <- st_obj@meta.data[spot_ids, , drop = FALSE]

    required_columns <- c("x", "y", gt_cols)
    missing_columns <- setdiff(required_columns, colnames(st_meta))
    if (length(missing_columns) > 0) {stop("Missing columns in metadata for ", tp, ": ", paste(missing_columns, collapse = ", "))}
    
    coords <- st_meta[spot_ids, c("x", "y"), drop = FALSE]
    coords <- as.data.frame(coords)
    coords$x <- as.numeric(coords$x)
    coords$y <- as.numeric(coords$y)
    
    rownames(coords) <- spot_ids

    common_genes <- intersect(rownames(counts_sc), rownames(counts_st))
    if (length(common_genes) == 0) {stop("No shared genes found for timepoint: ", tp)}
    counts_sc <- counts_sc[common_genes, , drop = FALSE]
    counts_st <- counts_st[common_genes, , drop = FALSE]
    
    cat("Reference cells:", ncol(counts_sc), "| Spatial spots:", ncol(counts_st), "| Shared genes:", length(common_genes), "\n")

    scRNA <- Reference(counts = counts_sc, cell_types = cluster_sc, nUMI = nUMI_sc)
    stRNA <- SpatialRNA(coords = coords, counts = counts_st, nUMI = colSums(counts_st))

    invisible(capture.output({
        myRCTD <- create.RCTD(stRNA, scRNA, max_cores = 8)
        myRCTD <- suppressMessages(run.RCTD(myRCTD, doublet_mode = "full"))
    }))
    
    norm_weights <- normalize_weights(myRCTD@results$weights)
    norm_weights <- as.data.frame(norm_weights)
    norm_weights <- norm_weights[spot_ids, , drop = FALSE]

    for (ct in cell_types) {
        if (!ct %in% colnames(norm_weights)) {norm_weights[[ct]] <- 0}
    }

    norm_weights <- norm_weights[, cell_types, drop = FALSE]

    result_tp <- data.frame(timepoint = tp, spot_id = spot_ids, x = st_meta[spot_ids, "x"], y = st_meta[spot_ids, "y"],
                            norm_weights, st_meta[spot_ids, gt_cols, drop = FALSE], check.names = FALSE, row.names = NULL)

    rctd_standardized_results[[tp]] <- result_tp
    rctd_objects[[tp]] <- myRCTD
    cat("Running RCTD for", tp, "was successfully generated.\n")
}
RCTD_all_timepoints <- do.call(rbind, rctd_standardized_results)
rownames(RCTD_all_timepoints) <- NULL

saveRDS(RCTD_all_timepoints, file = file.path(deconv_dir, "RCTD_all_timepoints.rds"), compress = TRUE)
write.csv(RCTD_all_timepoints, file = file.path(deconv_dir, "RCTD_all_timepoints_prop.csv"), row.names = FALSE)
cat("\n✅ RCTD pipeline completed!\n")

RCTD Running for Tumor ...
Reference cells: 1973 | Spatial spots: 3291 | Shared genes: 1000 


Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 1972

process_cell_type_info: number of genes in reference: 1000

End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 19

get_de_genes: Epithelial cells found DE genes: 0

get_de_genes: Neutrophils found DE genes: 48

get_de_genes: OPCs found DE genes: 6

get_de_genes: Endothelial cells found DE genes: 52

get_de_genes: Proliferation found DE genes: 39

get_de_genes: total DE genes: 151

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 58

get_de_genes: Epithelial cells found DE genes: 17

get_de_genes: Neutrophils found DE genes: 94

get_de_genes: OPCs found DE genes: 30

get_de_genes: Endothelial cells found DE genes: 110

get_de_genes: Proliferation found DE genes: 88

get_de_genes: total DE genes: 341



Running RCTD for Tumor was successfully generated.
RCTD Running for 2wk ...
Reference cells: 2475 | Spatial spots: 3291 | Shared genes: 1000 


Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 2475

process_cell_type_info: number of genes in reference: 1000

End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 22

get_de_genes: Epithelial cells found DE genes: 1

get_de_genes: Neutrophils found DE genes: 53

get_de_genes: OPCs found DE genes: 9

get_de_genes: Endothelial cells found DE genes: 42

get_de_genes: Proliferation found DE genes: 38

get_de_genes: total DE genes: 146

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 61

get_de_genes: Epithelial cells found DE genes: 12

get_de_genes: Neutrophils found DE genes: 89

get_de_genes: OPCs found DE genes: 29

get_de_genes: Endothelial cells found DE genes: 82

get_de_genes: Proliferation found DE genes: 84

get_de_genes: total DE genes: 296



Running RCTD for 2wk was successfully generated.
RCTD Running for 24wk ...
Reference cells: 552 | Spatial spots: 3291 | Shared genes: 1000 


Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 552

process_cell_type_info: number of genes in reference: 1000

End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 30

get_de_genes: Epithelial cells found DE genes: 8

get_de_genes: Neutrophils found DE genes: 58

get_de_genes: OPCs found DE genes: 17

get_de_genes: Endothelial cells found DE genes: 45

get_de_genes: Proliferation found DE genes: 45

get_de_genes: total DE genes: 181

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: Astrocytes found DE genes: 76

get_de_genes: Epithelial cells found DE genes: 41

get_de_genes: Neutrophils found DE genes: 102

get_de_genes: OPCs found DE genes: 72

get_de_genes: Endothelial cells found DE genes: 111

get_de_genes: Proliferation found DE genes: 100

get_de_genes: total DE genes: 436



Running RCTD for 24wk was successfully generated.

✅ RCTD pipeline completed!


###### *SPOTlight*

In [5]:
cell_types <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")
gt_cols <- paste0("gt_", cell_types)

# Reference preparation (as matrix)
cat("Preparing reference...\n")

# Extract as matrix (SPOTlight needs matrix)
ref_counts <- as.matrix(GetAssayData(SimulatedSC, assay = "RNA", layer = "counts"))
ref_meta <- SimulatedSC@meta.data

groups <- as.character(ref_meta$cell_types)
names(groups) <- colnames(ref_counts)

cat(sprintf("Reference: %d genes x %d cells\n", nrow(ref_counts), ncol(ref_counts)))

# Find marker genes
cat("Finding marker genes...\n")
Idents(SimulatedSC) <- "cell_types"

markers <- tryCatch({FindAllMarkers(SimulatedSC, only.pos = TRUE, logfc.threshold = 0, min.pct = 0.1)}, error = function(e) {NULL})

if (is.null(markers) || nrow(markers) == 0) {
    cat("Using all genes as markers...\n")
    all_genes <- rownames(ref_counts)
    markers <- data.frame(gene = rep(all_genes, length(cell_types)), cluster = rep(cell_types, each = length(all_genes)), avg_log2FC = 1, stringsAsFactors = FALSE)    
}
cat(sprintf("Markers: %d rows\n", nrow(markers)))

# Run SPOTlight for each time point
spotlight_standardized <- list()

for (tp in names(SimulatedST)) {
    cat("\nRunning SPOTlight for", tp, "\n")
    st_obj <- SimulatedST[[tp]]
    
    # Extract spatial as matrix
    spatial_counts <- as.matrix(GetAssayData(st_obj, assay = "RNA", layer = "counts"))
    spatial_meta <- st_obj@meta.data
    cat(sprintf("Spatial: %d genes x %d spots\n", nrow(spatial_counts), ncol(spatial_counts)))
    
    start_time <- Sys.time()
    spotlight_obj <- tryCatch({
        SPOTlight(x = ref_counts, y = spatial_counts, groups = groups, mgs = markers, weight_id = "avg_log2FC", group_id = "cluster", gene_id = "gene")
    }, error = function(e) {
        cat("❌ Error:", e$message, "\n")
        return(NULL)
    })

    if (is.null(spotlight_obj)) next

    elapsed <- difftime(Sys.time(), start_time, units = "mins")
    cat(sprintf("  Completed in %.1f minutes\n", elapsed))

    # Extract proportions
    spotlight_props <- as.data.frame(spotlight_obj$mat)
    
    # Ensure all cell types present
    for (ct in cell_types) {
        if (!ct %in% colnames(spotlight_props)) {spotlight_props[[ct]] <- 0}
    }
    spotlight_props <- spotlight_props[, cell_types, drop = FALSE]

    # Ground truth
    gt_data <- spatial_meta[, gt_cols, drop = FALSE]
    #colnames(gt_data) <- cell_types

    # Combine
    result_tp <- data.frame(timepoint = tp, spot_id = rownames(spatial_meta),
                            x = spatial_meta$x, y = spatial_meta$y, spotlight_props, gt_data, check.names = FALSE, row.names = NULL)
    
    spotlight_standardized[[tp]] <- result_tp
    cat("✅", tp, "Completed.\n")
}

# Save
if (length(spotlight_standardized) > 0) {
    SPOTlight_all_timepoints <- do.call(rbind, spotlight_standardized)
    rownames(SPOTlight_all_timepoints) <- NULL

    saveRDS(SPOTlight_all_timepoints, file.path(deconv_dir, "SPOTlight_all_timepoints.rds"), compress = TRUE)
    write.csv(SPOTlight_all_timepoints, file.path(deconv_dir, "SPOTlight_all_timepoints_prop.csv"), row.names = FALSE)
    
    cat("\n✅ SPOTlight pipeline completed!\n")
}

Preparing reference...
Reference: 1000 genes x 5000 cells
Finding marker genes...


Calculating cluster Astrocytes

Calculating cluster Epithelial cells

Calculating cluster Neutrophils

Calculating cluster OPCs

Calculating cluster Endothelial cells

Calculating cluster Proliferation



Using all genes as markers...
Markers: 6000 rows

Running SPOTlight for Tumor 
Spatial: 1000 genes x 3291 spots


Scaling count matrix

Seeding initial matrices

Training NMF model

Time for training: 25.68min

Deconvoluting mixture data



  Completed in 25.7 minutes
✅ Tumor Completed.

Running SPOTlight for 2wk 
Spatial: 1000 genes x 3291 spots


Scaling count matrix

Seeding initial matrices

Training NMF model

Time for training: 24.54min

Deconvoluting mixture data



  Completed in 24.6 minutes
✅ 2wk Completed.

Running SPOTlight for 24wk 
Spatial: 1000 genes x 3291 spots


Scaling count matrix

Seeding initial matrices

Training NMF model

Time for training: 25.04min

Deconvoluting mixture data



  Completed in 25.1 minutes
✅ 24wk Completed.

✅ SPOTlight pipeline completed!


###### *SpatialDecon*

In [6]:
get_expr_matrix <- function(seu) {as.matrix(GetAssayData(seu, assay = DefaultAssay(seu), layer = "counts"))}

make_profile_matrix <- function(sc_seu) {
    expr <- get_expr_matrix(sc_seu)
    cell_types <- as.character(sc_seu$cell_types)

    prof_list <- lapply(sort(unique(cell_types)), function(ct) {
        idx <- which(cell_types == ct)
        sub_expr <- expr[, idx, drop = FALSE]
        if (ncol(sub_expr) == 1) return(as.numeric(sub_expr))
        return(rowMeans(sub_expr, na.rm = TRUE))
    })

    prof <- do.call(cbind, prof_list)
    colnames(prof) <- sort(unique(cell_types))
    rownames(prof) <- rownames(expr)
    prof
}

run_spatialdecon_one_tp <- function(sc_seu, st_seu, tp_name) {
    profile_mtx <- make_profile_matrix(sc_seu)
    st_expr <- get_expr_matrix(st_seu)

    common_genes <- intersect(rownames(profile_mtx), rownames(st_expr))
    profile_mtx <- profile_mtx[common_genes, , drop = FALSE]
    st_expr <- st_expr[common_genes, , drop = FALSE]

    # Nomalization
    libsize <- colSums(st_expr)
    libsize[libsize == 0] <- 1
    norm_expr <- sweep(st_expr, 2, libsize, "/") * 1e6

    # Background
    bg_mtx <- matrix(0.1, nrow = nrow(norm_expr), ncol = ncol(norm_expr), dimnames = dimnames(norm_expr))

    res <- tryCatch({
        spatialdecon(norm = norm_expr, bg = bg_mtx, X = profile_mtx)
    }, error = function(e) {
        message("   Error in spatialdecon for ", tp_name, ": ", e$message)
        return(NULL)
    })

    if (is.null(res)) return(NULL)

    est <- if (!is.null(res$prop.of.all)) res$prop.of.all else res$beta
    est <- as.matrix(est)

    if (ncol(est) == ncol(st_seu)) est <- t(est)
    
    est <- est / rowSums(est)
    est[is.na(est)] <- 0

    meta <- st_seu@meta.data
    gt_cols <- grep("^gt_", colnames(meta), value = TRUE)

    gt_data <- meta[, gt_cols, drop = FALSE]

    out <- data.frame(timepoint = tp_name, spot_id = rownames(meta), x = meta$x, y = meta$y, check.names = FALSE, row.names = NULL)

    est_df <- as.data.frame(est)
    est_df$spot_id <- rownames(est)
    out <- merge(out, est_df, by = "spot_id", all.x = TRUE)

    gt_data$spot_id <- rownames(gt_data)
    out <- merge(out, gt_data, by = "spot_id", all.x = TRUE)
    
    return(out)
}

timepoints <- names(SimulatedST)
results_list <- list()

for (tp in timepoints) {
    message("Processing timepoint: ", tp)
    tp_res <- run_spatialdecon_one_tp(SimulatedSC, SimulatedST[[tp]], tp)
    if (!is.null(tp_res)) results_list[[tp]] <- tp_res
}

if (length(results_list) > 0) {
    final_df <- bind_rows(results_list)
    saveRDS(final_df, file.path(deconv_dir, "SpatialDecon_all_timepoints.rds"))
    write.csv(final_df, file.path(deconv_dir, "SpatialDecon_all_timepoints_prop.csv"), row.names = FALSE)
    message("✅ SpatialDecon pipeline completed!")
}

Processing timepoint: Tumor

Processing timepoint: 2wk

Processing timepoint: 24wk

✅ SpatialDecon pipeline completed!



###### *SpaOTsc*

In [3]:
cell_types <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")
gt_cols <- paste0("gt_", cell_types)

use_python("C:/Users/PURSAL~1/AppData/Local/Programs/Python/PYTHON~2/python.exe", required = TRUE)
source_python("spaotsc_r.py")

# Run SpaOTsc
run_spaotsc_tp <- function(sc_seu, st_obj, tp_name, cell_types) {
    cat("  Processing", tp_name, "...\n")
    
    sc_counts <- as.matrix(GetAssayData(sc_seu, assay = "RNA", layer = "counts"))
    sc_labels <- as.character(sc_seu$cell_types)
    
    st_counts <- as.matrix(GetAssayData(st_obj, assay = "RNA", layer = "counts"))
    st_coords <- st_obj@meta.data[, c("x", "y")]
    st_meta <- st_obj@meta.data
    n_spots <- ncol(st_counts)
    
    common_genes <- intersect(rownames(sc_counts), rownames(st_counts))
    sc_counts <- sc_counts[common_genes, ]
    st_counts <- st_counts[common_genes, ]
    
    cat("    Genes:", length(common_genes), "| Cells:", ncol(sc_counts), "| Spots:", n_spots, "\n")
    
    # Transfer to Python
    py$sc_data <- r_to_py(sc_counts)
    py$sc_labels <- r_to_py(sc_labels)
    py$st_data <- r_to_py(st_counts)
    py$st_coords <- r_to_py(as.matrix(st_coords))
    
    py_run_string("
result = run_spaotsc(sc_data, sc_labels, st_data, st_coords)
proportions = result[0]
ct_names = result[1]
")
    
    # Extract results
    proportions <- tryCatch(py$proportions, error = function(e) NULL)
    ct_names <- tryCatch(py$ct_names, error = function(e) NULL)
    
    if (is.null(proportions) || is.null(ct_names)) {
        cat("    ❌ Failed to get results\n")
        return(NULL)
    }
    
    props <- as.data.frame(matrix(0, nrow = n_spots, ncol = length(cell_types)))
    colnames(props) <- cell_types
    
    for (i in seq_along(ct_names)) {
        ct <- ct_names[i]
        if (ct %in% cell_types) {
            props[[ct]] <- proportions[, i]
        }
    }
    
    row_sums <- rowSums(props)
    props <- props / pmax(row_sums, 1e-10)
    props[is.na(props)] <- 0
    
    result_tp <- data.frame(timepoint = tp_name, spot_id = rownames(st_meta), x = st_meta$x, y = st_meta$y, stringsAsFactors = FALSE)
    
    for (ct in cell_types) {result_tp[[ct]] <- props[[ct]]}
    
    for (gt_col in gt_cols) {result_tp[[gt_col]] <- st_meta[[gt_col]]}
    
    rownames(result_tp) <- NULL
    cat("    ✅ Done\n")
    return(result_tp)
}

cat("\n========================================\n")
cat("STARTING SpaOTsc Analysis\n")
cat("========================================\n")

spaotsc_results <- list()
for (tp in names(SimulatedST)) {
    cat("\n--- Timepoint:", tp, "---\n")
    
    result <- tryCatch(
        run_spaotsc_tp(SimulatedSC, SimulatedST[[tp]], tp, cell_types),
        error = function(e) {
            cat("  Error:", conditionMessage(e), "\n")
            NULL
        }
    )
    
    if (!is.null(result)) {spaotsc_results[[tp]] <- result}
}

# Save results
if (length(spaotsc_results) > 0) {
    spaotsc_all <- do.call(rbind, spaotsc_results)
    rownames(spaotsc_all) <- NULL
    
    saveRDS(spaotsc_all, file.path(deconv_dir, "spaotsc_all_timepoints.rds"))
    write.csv(spaotsc_all, file.path(deconv_dir, "spaotsc_all_timepoints_prop.csv"), row.names = FALSE)
    
    cat("\n✅ SpaOTsc pipeline completed successfully!\n")
    cat("Results saved to:", deconv_dir, "\n")
    
    # Validation
    cat("\n=== Validation Results ===\n")
    for (tp in unique(spaotsc_all$timepoint)) {
        tp_data <- spaotsc_all[spaotsc_all$timepoint == tp, ]
        
        pred_mat <- as.matrix(tp_data[, cell_types, drop = FALSE])
        gt_mat <- as.matrix(tp_data[, gt_cols, drop = FALSE])
        colnames(gt_mat) <- cell_types
        
        corr <- cor(as.vector(pred_mat), as.vector(gt_mat), use = "complete.obs")
        rmse <- sqrt(mean((pred_mat - gt_mat)^2, na.rm = TRUE))
        
        cat(sprintf("  %s: Correlation = %.4f, RMSE = %.4f\n", tp, corr, rmse))
    }
} else {
    cat("\n❌ SpaOTsc pipeline failed\n")
}


STARTING SpaOTsc Analysis

--- Timepoint: Tumor ---
  Processing Tumor ...
    Genes: 1000 | Cells: 5000 | Spots: 3291 
    ✅ Done

--- Timepoint: 2wk ---
  Processing 2wk ...
    Genes: 1000 | Cells: 5000 | Spots: 3291 
    ✅ Done

--- Timepoint: 24wk ---
  Processing 24wk ...
    Genes: 1000 | Cells: 5000 | Spots: 3291 
    ✅ Done

✅ SpaOTsc pipeline completed successfully!
Results saved to: ./Final_outputs/Deconvolution 

=== Validation Results ===
  Tumor: Correlation = 0.0911, RMSE = 0.2590
  2wk: Correlation = 0.0778, RMSE = 0.2619
  24wk: Correlation = 0.0575, RMSE = 0.2594


###### *CellTrek*

In [3]:
cell_types <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")
gt_cols <- paste0("gt_", cell_types)

# Preparing reference single-cell data...
cat("📊 Preparing reference single-cell data...\n")

if("cell_types" %in% colnames(SimulatedSC@meta.data)) {colnames(SimulatedSC@meta.data)[colnames(SimulatedSC@meta.data) == "cell_types"] <- "cell_type"}

invisible(capture.output({suppressMessages({suppressWarnings({sc_seurat <- SimulatedSC
                                                              sc_seurat <- NormalizeData(sc_seurat)
                                                              sc_seurat <- FindVariableFeatures(sc_seurat, nfeatures = nrow(sc_seurat))
                                                              sc_seurat <- ScaleData(sc_seurat)
                                                              sc_seurat <- RunPCA(sc_seurat, npcs = 30, verbose = FALSE)})})}))

cat("  - Reference cells:", ncol(sc_seurat))
cat("\n  - Genes:", nrow(sc_seurat))

celltrek_standardized_results <- list()
for (tp in names(SimulatedST)) {
    cat("\n🔬 Running CellTrek for", tp, "...\n")
    
    st_seurat <- SimulatedST[[tp]]
    
    # Normalization
    invisible(capture.output({suppressMessages({suppressWarnings({st_seurat <- NormalizeData(st_seurat)
                                                              st_seurat <- FindVariableFeatures(st_seurat, nfeatures = nrow(st_seurat))
                                                              st_seurat <- ScaleData(st_seurat)
                                                              st_seurat <- RunPCA(st_seurat, npcs = 30)})})}))
    
    common_genes <- intersect(rownames(sc_seurat), rownames(st_seurat))
    sc_sub <- sc_seurat[common_genes, ]
    st_sub <- st_seurat[common_genes, ]

    integration_features <- suppressMessages(VariableFeatures(sc_seurat))

    # Finding transfer anchors
    anchors <- suppressMessages(FindTransferAnchors(reference = sc_sub, query = st_sub, features = integration_features, reduction = "pcaproject", 
                                                    dims = 1:min(30, ncol(sc_sub) - 1, ncol(st_sub) - 1), k.anchor = 5, k.filter = 200))
    
    predictions <- TransferData(anchorset = anchors, refdata = sc_sub$cell_type, 
                                dims = 1:min(30, ncol(sc_sub) - 1, ncol(st_sub) - 1), weight.reduction = "pcaproject")
    st_sub <- AddMetaData(st_sub, predictions)

    n_spots <- ncol(st_sub)
    n_types <- length(cell_types)

    hard_props <- matrix(0, nrow = n_spots, ncol = n_types)
    colnames(hard_props) <- cell_types
    rownames(hard_props) <- colnames(st_sub)

    for(i in 1:n_spots) {
        pred_type <- as.character(st_sub$predicted.id[i])
        if(!is.na(pred_type) && pred_type %in% cell_types) {hard_props[i, pred_type] <- 1}
    }

    # Soft prediction با Weighted KNN
    sc_pca <- Embeddings(sc_sub, reduction = "pca")[, 1:min(30, ncol(sc_sub)), drop = FALSE]
    st_pca <- Embeddings(st_sub, reduction = "pca")[, 1:min(30, ncol(st_sub)), drop = FALSE]

    n_dims <- min(ncol(sc_pca), ncol(st_pca), 30)
    sc_pca <- sc_pca[, 1:n_dims, drop = FALSE]
    st_pca <- st_pca[, 1:n_dims, drop = FALSE]
    
    k_neighbors <- min(20, nrow(sc_pca))

    soft_props <- matrix(0, nrow = n_spots, ncol = n_types)
    colnames(soft_props) <- cell_types
    rownames(soft_props) <- colnames(st_sub)

    for(i in 1:n_spots) {
        query_vec <- st_pca[i, ]
        diff <- sc_pca - matrix(query_vec, nrow = nrow(sc_pca), ncol = n_dims, byrow = TRUE)
        distances <- sqrt(rowSums(diff^2))
        
        k_indices <- order(distances)[1:min(k_neighbors, length(distances))]
        weights <- 1 / (distances[k_indices] + 1e-10)
        weights <- weights / sum(weights)

        k_types <- as.character(sc_sub$cell_type[k_indices])
        for(j in 1:length(k_types)) {soft_props[i, k_types[j]] <- soft_props[i, k_types[j]] + weights[j]}
    }

    soft_props <- soft_props / rowSums(soft_props)
    
    final_props <- (hard_props * 0.3 + soft_props * 0.7)
    final_props <- final_props / rowSums(final_props)

    for (ct in cell_types) {
        if (!ct %in% colnames(final_props)) {final_props <- cbind(final_props, setNames(data.frame(rep(0, n_spots)), ct))}
    }
    final_props <- final_props[, cell_types, drop = FALSE]

    spot_ids <- colnames(st_sub)
    st_meta <- st_sub@meta.data[spot_ids, , drop = FALSE]

    if("spatial" %in% names(st_sub@reductions)) {
        coords <- Embeddings(st_sub, reduction = "spatial")
        rownames(coords) <- spot_ids
    } else {
        coords <- st_meta[, c("x", "y"), drop = FALSE]
    }

    result_tp <- data.frame(timepoint = rep(tp, n_spots), spot_id = spot_ids, x = coords[, 1], y = coords[, 2],
                            final_props, st_meta[, gt_cols, drop = FALSE], check.names = FALSE, row.names = NULL)

    celltrek_standardized_results[[tp]] <- result_tp
    cat("✅ CellTrek for", tp, "was successfully generated.\n")
}

# Save results
CellTrek_all_timepoints <- do.call(rbind, celltrek_standardized_results)
rownames(CellTrek_all_timepoints) <- NULL

cat("  - Total spots:", nrow(CellTrek_all_timepoints))
cat("\n  - Timepoints:", paste(unique(CellTrek_all_timepoints$timepoint), collapse = ", "))

saveRDS(CellTrek_all_timepoints, file = file.path(deconv_dir, "CellTrek_all_timepoints.rds"), compress = TRUE)
write.csv(CellTrek_all_timepoints, file = file.path(deconv_dir, "CellTrek_all_timepoints_prop.csv"), row.names = FALSE)
cat("\n✅ CellTrek pipeline completed!\n")

📊 Preparing reference single-cell data...
  - Reference cells: 5000
  - Genes: 1000
🔬 Running CellTrek for Tumor ...


Finding integration vectors

Finding integration vector weights

Predicting cell labels



✅ CellTrek for Tumor was successfully generated.

🔬 Running CellTrek for 2wk ...


Finding integration vectors

Finding integration vector weights

Predicting cell labels



✅ CellTrek for 2wk was successfully generated.

🔬 Running CellTrek for 24wk ...


Finding integration vectors

Finding integration vector weights

Predicting cell labels



✅ CellTrek for 24wk was successfully generated.
  - Total spots: 9873
  - Timepoints: Tumor, 2wk, 24wk
✅ CellTrek pipeline completed!


###### *Evaluation deconvolution methods*

In [4]:
deconv_plots <- "./Final_outputs/Deconvolution/Plots"
dir.create(deconv_plots, recursive = TRUE, showWarnings = FALSE)

# Load data
res_spatialdecon <- readRDS("./Final_outputs/Deconvolution/SpatialDecon_all_timepoints.rds")
res_rctd         <- readRDS("./Final_outputs/Deconvolution/RCTD_all_timepoints.rds")
res_card         <- readRDS("./Final_outputs/Deconvolution/CARD_all_timepoints.rds")
res_spotlight    <- readRDS("./Final_outputs/Deconvolution/SPOTlight_all_timepoints.rds")
res_spaotsc      <- readRDS("./Final_outputs/Deconvolution/spaotsc_all_timepoints.rds")
res_celltrek     <- readRDS("./Final_outputs/Deconvolution/CellTrek_all_timepoints.rds")

cell_types <- c("Astrocytes", "Epithelial cells", "Neutrophils", "OPCs", "Endothelial cells", "Proliferation")
methods <- c("SpatialDecon", "RCTD", "CARD", "SPOTlight", "SpaOTsc", "CellTrek")
timepoints <- c("Tumor", "2wk", "24wk")
method_colors <- c("RCTD" = "#E41A1C", "CARD" = "#377EB8", "SpatialDecon" = "#4DAF4A", "SPOTlight" = "#984EA3", "SpaOTsc" = "#FF7F00", "CellTrek" = "#00BFC4")

# Calculate Metrics
cat("Calculating metrics...\n")
metrics_list <- list()

all_res <- list(res_spatialdecon, res_rctd, res_card, res_spotlight, res_spaotsc, res_celltrek)
all_names <- c("SpatialDecon", "RCTD", "CARD", "SPOTlight", "SpaOTsc", "CellTrek")

for (idx in seq_along(all_res)) {
    res_obj <- all_res[[idx]]
    method_name <- all_names[idx]
    
    for (tp in timepoints) {
        tp_data <- res_obj[res_obj$timepoint == tp, ]
        if (nrow(tp_data) == 0) next
        for (ct in cell_types) {
            gt_col <- paste0("gt_", ct)
            pred <- tp_data[[ct]]
            gt <- tp_data[[gt_col]]

            rmse <- sqrt(mean((pred - gt)^2, na.rm = TRUE))
            pcc <- tryCatch(cor(pred, gt, method = "pearson"), error = function(e) NA)
                            mae <- mean(abs(pred - gt), na.rm = TRUE)

            # JSD
            eps <- 1e-10
            p_norm <- (pred + eps) / sum(pred + eps)
            g_norm <- (gt + eps) / sum(gt + eps)
            m_norm <- (p_norm + g_norm) / 2
            jsd <- 0.5 * sum(p_norm * log(p_norm / m_norm)) + 0.5 * sum(g_norm * log(g_norm / m_norm))
            metrics_list[[length(metrics_list) + 1]] <- data.frame(Method = method_name, Timepoint = tp, CellType = ct,
                                                                   RMSE = rmse, PCC = pcc, MAE = mae, JSD = jsd, stringsAsFactors = FALSE)
        }
    }
}
metrics_df <- do.call(rbind, metrics_list)

# Summaries
ct_summary <- metrics_df %>% group_by(Method, CellType) %>% summarise(Mean_PCC = mean(PCC, na.rm = TRUE), .groups = "drop")
summary_metrics <- metrics_df %>% group_by(Method, Timepoint) %>% summarise(Mean_RMSE = mean(RMSE, na.rm = TRUE), Mean_PCC = mean(PCC, na.rm = TRUE),
                                                                            Mean_MAE = mean(MAE, na.rm = TRUE), Mean_JSD = mean(JSD, na.rm = TRUE), .groups = "drop")

# Save CSVs
write.csv(metrics_df, file.path(deconv_plots, "metrics_full.csv"), row.names = FALSE)
write.csv(ct_summary, file.path(deconv_plots, "metrics_celltype_summary.csv"), row.names = FALSE)
write.csv(summary_metrics, file.path(deconv_plots, "metrics_summary.csv"), row.names = FALSE)
                            
# Build combined data for scatter/spatial
cat("Building combined data...\n")
combine_data <- function(data, method_name) {
    df <- data.frame(timepoint = data$timepoint, spot_id = data$spot_id, x = data$x, y = data$y, method = method_name)
    for (ct in cell_types) {
        gt_col <- paste0("gt_", ct)
        df[[paste0("pred_", ct)]] <- data[[ct]]
        df[[paste0("gt_", ct)]] <- data[[gt_col]]
    }
    df
}
all_data <- rbind(combine_data(res_spatialdecon, "SpatialDecon"), combine_data(res_rctd, "RCTD"), combine_data(res_card, "CARD"), 
                  combine_data(res_spotlight, "SPOTlight"), combine_data(res_spaotsc, "SpaOTsc"), combine_data(res_celltrek, "CellTrek"))

# Long format
long_list <- list()
for (ct in cell_types) {
    pred_col <- paste0("pred_", ct)
    gt_col <- paste0("gt_", ct)
    temp <- all_data[, c("timepoint", "spot_id", "x", "y", "method", pred_col, gt_col)]
    names(temp)[names(temp) == pred_col] <- "pred"
    names(temp)[names(temp) == gt_col] <- "gt"
    temp$cell_type <- ct
    long_list[[ct]] <- temp
}
long_data <- do.call(rbind, long_list)
rownames(long_data) <- NULL

# COMBINED OVERVIEW (PCC + RMSE)
cat("Creating Combined Overview (PCC & RMSE)...\n")
p_pcc <- ggplot(metrics_df, aes(x = Method, y = PCC, fill = Method)) + geom_boxplot(alpha = 0.7, outlier.shape = NA) +
                            geom_jitter(width = 0.15, size = 1, alpha = 0.3, aes(color = Method)) + theme_minimal(base_size = 14) +
                            labs(title = "Pearson Correlation (PCC)", y = "PCC", x = "") + scale_fill_manual(values = method_colors) +
                            scale_color_manual(values = method_colors) + 
                            theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1))
                            
p_rmse <- ggplot(metrics_df, aes(x = Method, y = RMSE, fill = Method)) + geom_boxplot(alpha = 0.7, outlier.shape = NA) +
                            geom_jitter(width = 0.15, size = 1, alpha = 0.3, aes(color = Method)) + theme_minimal(base_size = 14) +
                            labs(title = "Root Mean Square Error (RMSE)", y = "RMSE", x = "") + scale_fill_manual(values = method_colors) +
                            theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1))
P1 <- ggarrange(p_pcc, p_rmse, ncol = 2, nrow = 1)
ggsave(file.path(deconv_plots, "PCC_RMSE.png"), P1, width = 14, height = 5)
                            
# HEATMAP CELLTYPE × METHOD
cat("Creating Heatmap CellType × Method...\n")
P2 <- ct_summary %>% ggplot(aes(x = Method, y = CellType, fill = Mean_PCC)) + geom_tile(color = "white", size = 1) +
                            geom_text(aes(label = round(Mean_PCC, 3)), size = 3.5) + 
                            scale_fill_gradient2(low = "#D73027", mid = "white", high = "#4575B4", midpoint = 0.5, limits = c(0, 1)) +
                            theme_minimal(base_size = 14) + labs(title = "Mean PCC: Cell Type × Method", fill = "PCC") +
                            theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 11), axis.text.y = element_text(size = 12))
ggsave(file.path(deconv_plots, "heatmap_celltype.png"), P2, width = 10, height = 6)
                            
# SCATTER PREDICTED vs GROUND TRUTH
cat("Creating Scatter Predicted vs Ground Truth Proportions...\n")
long_data$timepoint <- factor(long_data$timepoint, levels = c("Tumor", "2wk", "24wk"))
long_data$cell_type <- factor(long_data$cell_type, levels = cell_types)
long_data$method <- factor(long_data$method, levels = c("RCTD", "CARD", "SpatialDecon", "SPOTlight", "SpaOTsc", "CellTrek"))
P3 <- long_data %>% ggplot(aes(x = gt, y = pred)) + geom_point(alpha = 0.1, size = 0.3, color = "steelblue") +
                            geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 0.5) + facet_grid(method ~ timepoint + cell_type) +
                            theme_minimal(base_size = 8) + 
                            labs(title = "Predicted vs Ground Truth Proportions", x = "Ground Truth", y = "Predicted") + coord_fixed() +
                            scale_x_continuous(limits = c(0, 1), breaks = c(0, 0.5, 1)) + scale_y_continuous(limits = c(0, 1), breaks = c(0, 0.5, 1)) +
                            theme(panel.spacing = unit(0.2, "lines"), strip.text = element_text(size = 6))

ggsave(file.path(deconv_plots, "scatter_pred_vs_gt.png"), P3, width = 20, height = 12)
                            
# SPATIAL COMPARISON
cat("Creating Spatial Comparison...\n")
create_spatial_plot <- function(tp, ct, method_name = NULL) {
    if (is.null(method_name)) {
        data <- all_data[all_data$timepoint == tp, ]
        col_name <- paste0("gt_", ct)
        title <- "Ground Truth"
    } else {
        data <- all_data[all_data$timepoint == tp & all_data$method == method_name, ]
        col_name <- paste0("pred_", ct)
        title <- method_name
    }

    ggplot(data, aes(x = x, y = y, color = .data[[col_name]])) + geom_point(size = 0.6) + 
    scale_color_gradientn(colors = c("navy", "white", "red"), limits = c(0, 1), oob = squish) + theme_void() +
    labs(title = title, color = "") + theme(plot.title = element_text(size = 8, face = "bold", hjust = 0.5), legend.position = "right", legend.key.size = unit(0.3, "cm"))
}

p_gt <- create_spatial_plot("Tumor", "Astrocytes")
p_rctd <- create_spatial_plot("Tumor", "Astrocytes", "RCTD")
p_card <- create_spatial_plot("Tumor", "Astrocytes", "CARD")
p_sd <- create_spatial_plot("Tumor", "Astrocytes", "SpatialDecon")
p_sl <- create_spatial_plot("Tumor", "Astrocytes", "SPOTlight")
p_sp <- create_spatial_plot("Tumor", "Astrocytes", "SpaOTsc")
p_celltrek <- create_spatial_plot("Tumor", "Astrocytes", "CellTrek")

P4 <- (p_gt | p_rctd | p_card | p_sd | p_sl | p_sp | p_celltrek) + plot_annotation(title = "Spatial Distribution: Astrocytes in Tumor",
                                                                                   theme = theme(plot.title = element_text(size = 14, face = "bold")))

ggsave(file.path(deconv_plots, "spatial_comparison.png"), P4, width = 24, height = 5)

# FINAL RANKING
cat("\n")
cat("========================================\n")
cat("         FINAL RANKING\n")
cat("========================================\n\n")

ranking <- summary_metrics %>% group_by(Method) %>% summarise(Mean_PCC = mean(Mean_PCC, na.rm = TRUE), Mean_RMSE = mean(Mean_RMSE, na.rm = TRUE),
                                                              Mean_MAE = mean(Mean_MAE, na.rm = TRUE), Mean_JSD = mean(Mean_JSD, na.rm = TRUE),
                                                              .groups = "drop") %>% arrange(desc(Mean_PCC))
                            
cat(sprintf("%-4s %-14s %-10s %-10s %-10s %-10s\n", "Rank", "Method", "PCC", "RMSE", "MAE", "JSD"))
cat(paste(rep("-", 55), collapse = ""), "\n")

for (i in 1:nrow(ranking)) {
    cat(sprintf("%-4d %-14s %-10.4f %-10.4f %-10.4f %-10.4f\n", i, ranking$Method[i], ranking$Mean_PCC[i], 
                ranking$Mean_RMSE[i], ranking$Mean_MAE[i], ranking$Mean_JSD[i]))
}

cat("\n========================================\n")
cat("✅ ALL DONE!\n")
cat("========================================\n")
cat("Plots:  ", deconv_plots, "\n")
cat("Metrics: ", deconv_plots, "\n")

Calculating metrics...
Building combined data...
Creating Combined Overview (PCC & RMSE)...
Creating Heatmap CellType × Method...
Creating Scatter Predicted vs Ground Truth Proportions...
Creating Spatial Comparison...

         FINAL RANKING

Rank Method         PCC        RMSE       MAE        JSD       
------------------------------------------------------- 
1    RCTD           0.7778     0.1225     0.0843     0.0883    
2    SpatialDecon   0.7688     0.1301     0.0976     0.1167    
3    CARD           0.6200     0.1371     0.1037     0.1147    
4    SPOTlight      0.1964     0.2073     0.1641     0.2354    
5    CellTrek       0.1540     0.2427     0.1776     0.2839    
6    SpaOTsc        0.0924     0.2539     0.1720     0.6010    

✅ ALL DONE!
Plots:   ./Final_outputs/Deconvolution/Plots 
Metrics:  ./Final_outputs/Deconvolution/Plots 


#### **Evaluation**